# Symbolic calculus : 

In [15]:
import sympy as sp
import numpy as np

from IPython.display import display


In [ ]:
b, m, x, i, eta = sp.symbols('b m x i eta', positive = True, real = True)
P = sp.Function("P")

def diff_and_multiply_symbolic(original,denominator, number_application = 1):
    '''
    This apply the operator we defined in the paper as $f$ : 
    - First divide by some polynome (here "denominator")
    - Then differentiate the result
    We do this "number_application" times and return a list with the result of each computation, multiplied by (-1)**order as it's its 
    sign in the final sum we will use it in.
    '''
    complete_terms = []
    rational_fraction = original
    for order in range(number_application):
        rational_fraction = rational_fraction/denominator
        complete_terms.append(sp.expand(((-1)**order)*rational_fraction))
        rational_fraction = sp.cancel(sp.factor(rational_fraction.diff()))
    return complete_terms

def derivative_Q(order):
    '''
    Automatically returns the derivative of Q with the form we calculated (can be done automatically by sympy but simplify some symbolic 
    manipulation)
    '''
    if order < 0 :
        raise ValueError("The order argument cannot be negative")
    elif order == 0 : 
        return (x*((m-x)**(b-1))/eta**(b-1))
    elif order == 1 :
        derivatives = (((m-x)**(b-2))/eta**(b-1)) * (b*x-m)
    else : 
        derivatives = sp.Product((b-i), (i,1,order-1))*((-1)**order) * (((m-x)**(b-order-1))/eta**(b-1)) * (order*m-b*x)
    return derivatives

def substitute_derivative(list_to_sub, order, polynomial = sp.Function("P"), get_derivative = derivative_Q):
    '''
    Substitutes the purely symbolic derivatives into their explicit form in the different component of 
    "list_to_sub"
    '''
    purely_symbolic_derivative = []
    explicit_form = []
    P_diff = polynomial(x)
    for derivative in range(order):
        P_diff = P_diff.diff()
        purely_symbolic_derivative.append(P_diff)
        explicit_form.append(get_derivative(derivative+1))
    for purely_symbolic, concrete_form in zip(reversed(purely_symbolic_derivative), reversed(explicit_form)):
        for index in range(len(list_to_sub)): 
            list_to_sub[index] = list_to_sub[index].subs(purely_symbolic,concrete_form)
    return list_to_sub

def evaluate (list_terms):
    '''
    Evaluates at 0 the different symbolic expression in "list_terms".
    '''
    for index in range(len(list_terms)):
        list_terms[index] = sp.factor(sp.powsimp(list_terms[index].subs(x,0)))
    return list_terms

def find_coeff_equivalent(original,denominator, number_application = 1, max_order_to_sub = 5, polynomial = sp.Function("P")):
    '''
    Apply Lemma A.8 to find the different coefficient for the equivalent 
    '''
    return evaluate(substitute_derivative(diff_and_multiply_symbolic(original, denominator, number_application = number_application), max_order_to_sub, polynomial = polynomial))

def develop_and_simplify_polynome(expr, collecting_variable, handleO = True, deep_expand = False, deep_factor = True):
    '''
    Develops and simplify a polynome. It use collecting variable as the variable with respect to which the simplification will happen. 
    Basically we consider all the power of the collecting variable and then simplify their coefficient one by one to recombine afterward. 


    Note : deep expand should be False if there is a denominator 
    '''
    expr = expr.expand(deep=deep_expand).powsimp()
    significativity_limit = 0
    if handleO : 
        significativity_limit = expr.getO()
        expr = expr.removeO()
    all_power_present = []
    for power in expr.atoms(sp.Pow) :
        power_as_base_exp = power.as_base_exp()
        if power_as_base_exp[0] == collecting_variable:
            all_power_present.append(power)

    expr = expr.collect(all_power_present)
    factorised_expr = 0
    for arg in expr.args:
        factorised_expr += arg.powsimp().factor(deep = deep_factor)

    return factorised_expr + significativity_limit

### Calculus of the building blocks : 

The following will be the different building blocks that we will use to compute the un-normalised expected value in the next part.

$$\int_0^{m-\eta}\exp(-g\left(\frac{m-g}{\eta}\right)^{b-1})dg$$

In [17]:
#First we calculate the parts of the sum and then we simplify it one by one.

P_diff = P(x).diff()
coeff_int_1 = -np.array(find_coeff_equivalent(1, P_diff, 4, 5, polynomial = P))
coeff_int_1 = np.array([coeff.simplify().factor() for coeff in list(coeff_int_1)])
display(list(coeff_int_1))
display(np.sum(coeff_int_1))

[eta**(b - 1)*m**(1 - b),
 2*eta**(2*b - 2)*m**(1 - 2*b)*(b - 1),
 3*eta**(3*b - 3)*m**(1 - 3*b)*(b - 1)*(3*b - 2),
 8*eta**(4*b - 4)*m**(1 - 4*b)*(b - 1)*(2*b - 1)*(4*b - 3)]

eta**(b - 1)*m**(1 - b) + 2*eta**(2*b - 2)*m**(1 - 2*b)*(b - 1) + 3*eta**(3*b - 3)*m**(1 - 3*b)*(b - 1)*(3*b - 2) + 8*eta**(4*b - 4)*m**(1 - 4*b)*(b - 1)*(2*b - 1)*(4*b - 3)

$$\int_0^{m-\eta}g\exp(-g\left(\frac{m-g}{\eta}\right)^{b-1})dg$$

In [18]:
# Same
coeff_int_x = -np.array(find_coeff_equivalent(x, P_diff, 4, 5))
display(np.sum([coeff.simplify().factor() for coeff in coeff_int_x]))
print(sp.latex(np.sum([coeff.simplify().factor() for coeff in coeff_int_x])))

eta**(2*b - 2)*m**(2 - 2*b) + 6*eta**(3*b - 3)*m**(2 - 3*b)*(b - 1) + 12*eta**(4*b - 4)*m**(2 - 4*b)*(b - 1)*(4*b - 3)

\eta^{2 b - 2} m^{2 - 2 b} + 6 \eta^{3 b - 3} m^{2 - 3 b} \left(b - 1\right) + 12 \eta^{4 b - 4} m^{2 - 4 b} \left(b - 1\right) \left(4 b - 3\right)


# Un-normalized expected value :

$$ \alpha\mathbb{E}(G|M>m) = \int_0^{m-\eta}g\exp(-g\left(\frac{m-g}{\eta}\right)^{b-1})dg + \int_{m-\eta}^{\infty}g\exp(-g) dg
$$

In [19]:
all_power_present = [m**(2-2*b), m**(2-3*b), m**(2-4*b)]
result_esp_g = np.sum(coeff_int_x)

result_esp_g = [result_esp_g.coeff(power).simplify().factor()*power for power in all_power_present]
display(np.sum(result_esp_g))

eta**(2*b - 2)*m**(2 - 2*b) + 6*eta**(3*b - 3)*m**(2 - 3*b)*(b - 1) + 12*eta**(4*b - 4)*m**(2 - 4*b)*(b - 1)*(4*b - 3)

$$\begin{align*}
    \alpha\mathbb{E}(\xi|M>m) =& \int_0^{m-\eta} (b-1)
    u\exp(-u\left(\frac{m-u}{\eta}\right)^{b-1}) du + \int_0^{m-\eta}(b-1)\frac{\eta^{b-1}}{(m-u)^{b-1}}\exp(-u\left(\frac{m-u}{\eta}\right)^{b-1}) du + \frac{(b-1)\eta^{b-1}}{(b-2)m^{b-2}}\\
    =&  \frac{(b-1)\eta^{b-1}}{(b-2)m^{b-2}} -\frac{\eta^{b-1}(b-1)}{m^{b-2}(b-2)} + \int_0^{m-\eta} (b-1)
    u\exp(-u\left(\frac{m-u}{\eta}\right)^{b-1}) du +\frac{b-1}{b-2}\int_0^{m-\eta}(m-bu)\exp(-u\left(\frac{m-u}{\eta}\right)^{b-1}) du + \mathcal{O}(\exp(-m))\\
    =&\int_0^{m-\eta} (b-1)u\exp(-u\left(\frac{m-u}{\eta}\right)^{b-1}) du +\frac{b-1}{b-2}\int_0^{m-\eta}(m-bu)\exp(-u\left(\frac{m-u}{\eta}\right)^{b-1}) du + \mathcal{O}(\exp(-m))
\end{align*}$$

In [20]:
result_esp_xi = coeff_int_x*(b-1) + coeff_int_1*(m*(b-1)/(b-2)) - coeff_int_x*(b*(b-1)/(b-2))
result_esp_xi = [coeff.powsimp().collect([m**(2-b),m**(2-4*b),m**(2-3*b),m**(2-2*b)]).simplify() for coeff in result_esp_xi]
result_esp_xi = [ result_esp_xi[index].coeff(power).simplify().factor()*power for index,power in enumerate([m**(2-b),m**(2-2*b),m**(2-3*b),m**(2-4*b)])]
display(result_esp_xi)
print(sp.latex(np.sum(result_esp_xi)))


[eta**(b - 1)*m**(2 - b)*(b - 1)/(b - 2),
 2*eta**(2*b - 2)*m**(2 - 2*b)*(b - 1),
 9*eta**(3*b - 3)*m**(2 - 3*b)*(b - 1)**2,
 16*eta**(4*b - 4)*m**(2 - 4*b)*(b - 1)**2*(4*b - 3)]

\frac{\eta^{b - 1} m^{2 - b} \left(b - 1\right)}{b - 2} + 2 \eta^{2 b - 2} m^{2 - 2 b} \left(b - 1\right) + 9 \eta^{3 b - 3} m^{2 - 3 b} \left(b - 1\right)^{2} + 16 \eta^{4 b - 4} m^{2 - 4 b} \left(b - 1\right)^{2} \left(4 b - 3\right)


# Normalisation : 

Here we first calculate an equivalent for $1/\alpha$ and then normalize the precedent computation.


#### Equivalent for $\frac{1}{\alpha}$

In [21]:
alpha = np.sum(coeff_int_1)
alpha = alpha.factor().powsimp()
#Find the common multiplicator and the thing it multiply : 
common_multiplicator = 1
coeff = 0
for value in alpha.args:
    if len(value.atoms(sp.Pow)) < 2:
        common_multiplicator *= value
    else : 
        coeff = value

#Note : here we have to verify the part which is constant to divide the square of the denominator by it and multiply the common 
#multiplicator by it : 
constante = 0
for part in coeff.args :
        if not(m in part.atoms()):
            constante += part

constante = constante.factor()
common_multiplicator = common_multiplicator*constante
coeff = (coeff/constante)

#substract 1 to the value of coeff (as the equivalent that will be applied is for x in 1/sqrt(1+x)) and here we have 1+x :

q_geom_serie = coeff - 1
q_geom_serie = develop_and_simplify_polynome(q_geom_serie, m, handleO=False)

#now we can replace the value in the limited development of 1/1+x and add the O (which we can add just like this 
# because of the 1/2 * x in the DL): 
dev_geometric_serie = 1 - x + x**2 - x**3
alpha_1 = develop_and_simplify_polynome(dev_geometric_serie.subs(x, q_geom_serie), m, deep_expand = True,deep_factor=True, handleO= False) 
alpha_display = alpha_1 + sp.Order(m**(-4*b),(m, sp.oo))
print(sp.latex(alpha_display))
#now we divide this by the square root of the common multiplicator : 
alpha_1 = develop_and_simplify_polynome(alpha_1/common_multiplicator,m, handleO=False) + sp.Order(m**(-3*b-1),(m, sp.oo))

print(sp.latex(alpha_1))
display(alpha_1)

1 - 4 \eta^{3 b - 3} m^{- 3 b} \left(b - 1\right) \left(3 b - 2\right) \left(3 b - 1\right) - \eta^{2 b - 2} m^{- 2 b} \left(b - 1\right) \left(5 b - 2\right) - 2 \eta^{b - 1} m^{- b} \left(b - 1\right) + O\left(m^{- 4 b}; m\rightarrow \infty\right)
- \frac{2 \left(b - 1\right)}{m} - 4 \eta^{2 b - 2} m^{- 2 b - 1} \left(b - 1\right) \left(3 b - 2\right) \left(3 b - 1\right) - \eta^{b - 1} m^{- b - 1} \left(b - 1\right) \left(5 b - 2\right) + \eta^{1 - b} m^{b - 1} + O\left(m^{- 3 b - 1}; m\rightarrow \infty\right)


-2*(b - 1)/m - 4*eta**(2*b - 2)*m**(-2*b - 1)*(b - 1)*(3*b - 2)*(3*b - 1) - eta**(b - 1)*m**(-b - 1)*(b - 1)*(5*b - 2) + eta**(1 - b)*m**(b - 1) + O(m**(-3*b - 1), (m, oo))

#### $\mathbb{E}(G|M>m)$

In [22]:
esp_g = np.sum(result_esp_g)*alpha_1
esp_g = develop_and_simplify_polynome(esp_g, m)
display(esp_g)

-2*eta**(4*b - 4)*m**(1 - 4*b)*(b - 1)*(3*b - 2)*(27*b - 23) + eta**(3*b - 3)*m**(1 - 3*b)*(b - 1)*(31*b - 22) + 4*eta**(2*b - 2)*m**(1 - 2*b)*(b - 1) + eta**(b - 1)*m**(1 - b) + O(m**(1 - 5*b), (m, oo))

#### $\mathbb{E}(\xi|M>m)$

In [23]:
esp_xi = np.sum(result_esp_xi)*alpha_1
esp_xi = develop_and_simplify_polynome(esp_xi, m)
display(esp_xi)

m*(b - 1)/(b - 2) - 2*eta**(3*b - 3)*m**(1 - 3*b)*(b - 1)**2*(31*b - 22)/(b - 2) - 8*eta**(2*b - 2)*m**(1 - 2*b)*(b - 1)**2/(b - 2) - 2*eta**(b - 1)*m**(1 - b)*(b - 1)/(b - 2) + O(m**(1 - 4*b), (m, oo))